In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieLensLSHSIgnature") \
    .getOrCreate()

sc = spark.sparkContext

26/03/02 21:52:04 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/02 21:52:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 21:52:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 21:52:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/02 21:52:06 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [5]:
from pyspark import SparkContext
import random

In [7]:
file_path = "/home/rajesh/CSL7100/Assignment2/MovieLens/ml-100k/u.data"

# Read file as RDD of lines
data = sc.textFile(file_path)

In [8]:
# Split each line and extract (user, movie)
user_movies = (
    data
    .map(lambda line: line.split('\t'))            # split each line
    .map(lambda x: (int(x[0]), int(x[1])))         # (user_id, movie_id)
    .groupByKey()                                  # group movies by user
    .mapValues(lambda movies: set(movies))         # convert to set (remove duplicates)
    .cache()                                       # cache for reuse
)

users = user_movies.keys().collect()               # list of user IDs

In [9]:
def true_jaccard(set1, set2):
    return float(len(set1 & set2)) / len(set1 | set2)   # |A∩B| / |A∪B|

In [10]:
# Create all user pairs using cartesian
user_pairs = (
    user_movies.cartesian(user_movies)
    .filter(lambda x: x[0][0] < x[1][0])   # avoid duplicate pairs
)

# Compute exact similarity
exact_pairs = (
    user_pairs
    .map(lambda x: (
        (x[0][0], x[1][0]),                     # (user1, user2)
        true_jaccard(x[0][1], x[1][1])          # similarity
    ))
    .filter(lambda x: x[1] >= 0.5)              # keep only >= 0.5
    .map(lambda x: x[0])                        # keep only user pair
)

exact_set = set(exact_pairs.collect())          # collect for comparison
print("Exact pairs >= 0.5:", len(exact_set))

[Stage 3:>                                                          (0 + 4) / 4]

Exact pairs >= 0.5: 10


In [20]:
import os
os.environ["PYTHONHASHSEED"] = "0"

In [21]:

def generate_hash_functions(num_hashes):

    p = 2003    # prime > 1682 (number of movies)

    hash_funcs = []
    for _ in range(num_hashes):
        a = random.randint(1, 1682)    # random coefficient
        b = random.randint(0, 1682)
        hash_funcs.append((a, b, p))

    return hash_funcs

In [22]:
def compute_signature(movie_set, hash_funcs):

    signature = []

    for (a, b, p) in hash_funcs:

        # Compute hash for every movie in user's set
        min_hash = min((a * m + b) % p for m in movie_set)

        signature.append(min_hash)   # store minimum hash value

    return signature

In [23]:
def estimate_jaccard(sig1, sig2):

    matches = 0

    for i in range(len(sig1)):
        if sig1[i] == sig2[i]:
            matches += 1

    return matches / float(len(sig1))   # fraction of matching positions

In [24]:
def compute_exact_pairs(threshold):

    user_pairs = (
        user_movies.cartesian(user_movies)
        .filter(lambda x: x[0][0] < x[1][0])   # avoid duplicate pairs
    )

    exact_pairs = (
        user_pairs
        .map(lambda x: (
            (x[0][0], x[1][0]),
            true_jaccard(x[0][1], x[1][1])
        ))
        .filter(lambda x: x[1] >= threshold)
        .map(lambda x: x[0])
    )

    return set(exact_pairs.collect())

In [25]:
import itertools
def run_lsh(num_hashes, r, b, threshold, runs=5):

    print("\nRunning LSH with:")
    print("Hashes =", num_hashes, " r =", r, " b =", b)
    print("Threshold =", threshold)

    exact_set = compute_exact_pairs(threshold)

    false_pos_total = 0
    false_neg_total = 0

    for run in range(runs):

        print("Run", run+1)

        # Generate hash functions
        hash_funcs = generate_hash_functions(num_hashes)
        bc_hash = sc.broadcast(hash_funcs)

        # Compute signatures
        signatures = (
            user_movies
            .mapValues(lambda movies: compute_signature(movies, bc_hash.value))
        )

        # -----------------------
        # LSH Banding
        # -----------------------

        def create_bands(user_sig):

            user_id, signature = user_sig
            bands = []

            for band in range(b):

                start = band * r
                end = start + r

                band_slice = tuple(signature[start:end])   # r values

                # Key = (band_number, band_signature)
                bands.append(((band, band_slice), user_id))

            return bands

        # Create (band_key, user) pairs
        banded = signatures.flatMap(create_bands)

        # Group users that share same band
        buckets = banded.groupByKey()

        # Generate candidate pairs from buckets
        candidate_pairs = (
            buckets
            .flatMap(lambda x: list(
                itertools.combinations(set(x[1]), 2)
            ))
            .distinct()
        )

        candidate_set = set(candidate_pairs.collect())

        # Compute actual similarity only for candidates
        approx_pairs = set()

        for (u1, u2) in candidate_set:

            set1 = user_movies.lookup(u1)[0]
            set2 = user_movies.lookup(u2)[0]

            sim = true_jaccard(set1, set2)

            if sim >= threshold:
                approx_pairs.add((u1, u2))

        # Compute errors
        false_pos = len(candidate_set - exact_set)
        false_neg = len(exact_set - candidate_set)

        false_pos_total += false_pos
        false_neg_total += false_neg

    print("Average False Positives:", false_pos_total / runs)
    print("Average False Negatives:", false_neg_total / runs)

In [ ]:
run_lsh(50, r=5, b=10, threshold=0.6)
run_lsh(100, r=5, b=20, threshold=0.6)
run_lsh(200, r=5, b=40, threshold=0.6)
run_lsh(200, r=10, b=20, threshold=0.6)


Running LSH with:
Hashes = 50  r = 5  b = 10
Threshold = 0.6


Run 1
Run 2
Run 3
Run 4
Run 5
Average False Positives: 478.0
Average False Negatives: 0.6

Running LSH with:
Hashes = 100  r = 5  b = 20
Threshold = 0.6


Run 1
Run 2


In [ ]:
run_lsh(50, r=5, b=10, threshold=0.8)
run_lsh(100, r=5, b=20, threshold=0.8)
run_lsh(200, r=5, b=40, threshold=0.8)
run_lsh(200, r=10, b=20, threshold=0.8)

In [3]:
if spark:
    spark.stop()

NameError: name 'spark' is not defined